# Chapter 5: AI Agents

Companion notebook for Chapter 5 of *Mastering AI-Powered Finance* (`Overleaf/Book 1`). It was built from Chapter 5 of the full manuscript (AI Agents); the source references below point to that manuscript.

Companion to `Overleaf/Final Project/Chapter 5 - AI Agents/main.tex`.

The three tools and worked example run offline. The final cell runs the agent only
when `RUN_LIVE_API=1` and `OPENAI_API_KEY` are set; it can incur API charges.
Install the repository requirements, including LangChain 1.x, before running.

This expanded notebook preserves the chapter's ReAct workflow while correcting the
credit-ratio definition and showing the tools separately so their results can be checked.


## Credit risk assessment tools

DTI is monthly debt payments divided by gross monthly income, as defined by the
[CFPB](https://www.consumerfinance.gov/ask-cfpb/what-is-a-debt-to-income-ratio-en-1791/).
Outstanding debt / annual income is retained as a
separately labelled leverage measure. The benchmark compares **proposed DTI**, which
includes the requested loan payment. The example adds an explicit, synthetic $600
existing monthly debt payment; that amount cannot be inferred from a $28,000 balance.

Living expenses exclude debt payments to prevent double counting. Because income
is gross, the coverage calculation is explicitly a **pre-tax proxy**, not disposable
income or a complete affordability assessment. Taxes, insurance, fees, and other
underwriting inputs are outside this teaching example. All sector thresholds,
default rates, and credit-band policies below are synthetic teaching inputs.

The current [LangChain agent API](https://docs.langchain.com/oss/python/migrate/langchain-v1)
uses `create_agent` and `system_prompt`. The live example keeps the chapter's `gpt-5`
model and omits `temperature`, which is not a supported tuning parameter for that
model's reasoning mode ([OpenAI documentation](https://developers.openai.com/api/docs/guides/latest-model?model=gpt-5.2#gpt-52-parameter-compatibility)).


In [1]:
import math
from typing import Annotated
from pydantic import Field
from langchain_core.tools import tool


@tool
def compute_financial_ratios(
    annual_income: Annotated[float, Field(gt=0, allow_inf_nan=False)],
    total_debt: Annotated[float, Field(ge=0, allow_inf_nan=False)],
    monthly_expenses: Annotated[float, Field(ge=0, allow_inf_nan=False)],
    monthly_debt_payments: Annotated[float, Field(ge=0, allow_inf_nan=False)],
    loan_amount: Annotated[float, Field(gt=0, allow_inf_nan=False)],
    loan_term_years: Annotated[int, Field(gt=0, strict=True)],
    interest_rate: Annotated[float, Field(ge=0, allow_inf_nan=False)],
) -> str:
    """Compute credit ratios for a synthetic loan application.

    annual_income is gross income; total_debt is outstanding principal.
    monthly_expenses excludes debt payments and the requested loan payment.
    monthly_debt_payments is existing monthly debt service, not debt balance.
    interest_rate is an annual percentage (6.5 means 6.5%). All monetary
    inputs use the same currency. Coverage is a pre-tax teaching proxy.
    """
    monthly_income = annual_income / 12
    monthly_rate = interest_rate / 100 / 12
    n_payments = loan_term_years * 12
    if monthly_rate > 0:
        # Stable even for very small positive rates or long loan terms.
        monthly_payment = loan_amount * monthly_rate / (
            -math.expm1(-n_payments * math.log1p(monthly_rate))
        )
    else:
        monthly_payment = loan_amount / n_payments

    existing_dti = monthly_debt_payments / monthly_income * 100
    proposed_dti = (monthly_debt_payments + monthly_payment) / monthly_income * 100
    balance_to_income = total_debt / annual_income * 100
    lti = loan_amount / annual_income
    pre_tax_buffer = monthly_income - monthly_expenses - monthly_debt_payments
    coverage_ratio = pre_tax_buffer / monthly_payment
    return (
        f"Existing Debt-to-Income (DTI): {existing_dti:.1f}%\n"
        f"Proposed DTI (including new loan): {proposed_dti:.1f}%\n"
        f"Existing Debt Balance / Annual Income (not DTI): {balance_to_income:.1f}%\n"
        f"Loan-to-Income (LTI): {lti:.2f}x\n"
        f"Estimated Monthly Payment: {monthly_payment:,.2f} currency units\n"
        f"Pre-Tax Payment Coverage Proxy: {coverage_ratio:.2f}x\n"
        f"Pre-Tax Buffer Before New Loan: {pre_tax_buffer:,.2f} currency units/month"
    )


@tool
def get_sector_benchmarks(sector: str) -> str:
    """Retrieve synthetic teaching thresholds, not calibrated lending policy.
    Sector options: 'retail', 'technology', 'manufacturing',
    'healthcare', 'hospitality'.
    DTI includes the new loan payment; coverage is a pre-tax teaching proxy."""
    benchmarks = {
        "retail": {
            "max_dti": "40%",
            "max_lti": "3.5x",
            "min_coverage": "1.3x",
            "default_rate": "3.2%",
            "risk_note": "Cyclical sector; seasonal income volatility common."
        },
        "technology": {
            "max_dti": "35%",
            "max_lti": "4.0x",
            "min_coverage": "1.5x",
            "default_rate": "1.8%",
            "risk_note": "Low default rates but high income concentration risk."
        },
        "manufacturing": {
            "max_dti": "38%",
            "max_lti": "3.0x",
            "min_coverage": "1.4x",
            "default_rate": "2.9%",
            "risk_note": "Sensitive to commodity prices and supply chain shocks."
        },
        "healthcare": {
            "max_dti": "36%",
            "max_lti": "3.8x",
            "min_coverage": "1.4x",
            "default_rate": "1.5%",
            "risk_note": "Stable sector with strong long-term income visibility."
        },
        "hospitality": {
            "max_dti": "32%",
            "max_lti": "2.5x",
            "min_coverage": "1.6x",
            "default_rate": "5.1%",
            "risk_note": "High cyclicality; tighter thresholds reflect elevated risk."
        },
    }
    data = benchmarks.get(sector.strip().lower())
    if not data:
        return (f"Sector '{sector}' not found. "
                f"Available: {', '.join(benchmarks.keys())}")
    return (
        "Illustrative synthetic benchmarks (not observed default rates).\n"
        f"Sector: {sector.strip().title()}\n"
        f"Maximum DTI: {data['max_dti']}\n"
        f"Maximum LTI: {data['max_lti']}\n"
        f"Minimum Pre-Tax Coverage Proxy: {data['min_coverage']}\n"
        f"Sector Default Rate: {data['default_rate']}\n"
        f"Risk Note: {data['risk_note']}"
    )


@tool
def lookup_credit_score_band(credit_score: Annotated[int, Field(strict=True)]) -> str:
    """Classify a credit score into a risk band and return
    the associated lending policy guidance.
    Input should be an integer between 300 and 850."""
    if not 300 <= credit_score <= 850:
        return "Invalid credit score. Must be between 300 and 850."
    if credit_score >= 800:
        band, policy = "Exceptional", "Eligible for preferential rates."
    elif credit_score >= 740:
        band, policy = "Very Good", "Standard approval; competitive rates."
    elif credit_score >= 670:
        band, policy = "Good", "Approve with standard terms."
    elif credit_score >= 580:
        band, policy = "Fair", "Approve with risk premium or reduced limit."
    elif credit_score >= 300:
        band, policy = "Poor", "Decline or require collateral and co-signer."
    else:
        return "Invalid credit score. Must be between 300 and 850."
    return (f"Credit Score: {credit_score} | Band: {band} | "
            f"Illustrative policy only: {policy}")




### Inspect the tools without an API call


In [2]:
# Offline worked example: all inputs are synthetic.
applicant = {
    "annual_income": 95_000,
    "total_debt": 28_000,
    "monthly_expenses": 3_200,
    "monthly_debt_payments": 600,
    "loan_amount": 320_000,
    "loan_term_years": 20,
    "interest_rate": 6.5,
}
print(compute_financial_ratios.invoke(applicant))
print()
print(get_sector_benchmarks.invoke({"sector": "technology"}))
print()
print(lookup_credit_score_band.invoke({"credit_score": 712}))


Existing Debt-to-Income (DTI): 7.6%
Proposed DTI (including new loan): 37.7%
Existing Debt Balance / Annual Income (not DTI): 29.5%
Loan-to-Income (LTI): 3.37x
Estimated Monthly Payment: 2,385.83 currency units
Pre-Tax Payment Coverage Proxy: 1.73x
Pre-Tax Buffer Before New Loan: 4,116.67 currency units/month

Illustrative synthetic benchmarks (not observed default rates).
Sector: Technology
Maximum DTI: 35%
Maximum LTI: 4.0x
Minimum Pre-Tax Coverage Proxy: 1.5x
Sector Default Rate: 1.8%
Risk Note: Low default rates but high income concentration risk.

Credit Score: 712 | Band: Good | Illustrative policy only: Approve with standard terms.


### Optional live agent

The prompt requests all three tools; the output check catches omitted or failed tools.
This is an observable tool-use demonstration, not a production underwriting control.


In [ ]:
import json
import os
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

if os.getenv("RUN_LIVE_API") != "1":
    print("Live agent skipped. Set RUN_LIVE_API=1 and OPENAI_API_KEY to run it.")
else:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY before enabling the live agent.")

    llm = ChatOpenAI(model="gpt-5", timeout=60, max_retries=2)
    agent = create_agent(
        model=llm,
        tools=[compute_financial_ratios, get_sector_benchmarks, lookup_credit_score_band],
        system_prompt=(
            "You are demonstrating credit analysis using synthetic teaching data. "
            "Always (1) compute financial ratios, (2) retrieve sector benchmarks, "
            "(3) look up the credit score band, and (4) provide a structured "
            "Approve / Decline / Escalate teaching recommendation with rationale. "
            "Compare proposed DTI, including the new loan, with the DTI threshold. "
            "Coverage is a pre-tax proxy, not verified disposable income. "
            "Do not infer missing inputs or treat synthetic default rates as evidence. "
            "Escalate missing or invalid data to a human reviewer. "
            "No tool approves a loan or executes an action."
        ),
    )
    application = "Assess this synthetic application:\n" + json.dumps(
        {**applicant, "sector": "technology", "credit_score": 712}, indent=2
    )
    result = agent.invoke(
        {"messages": [{"role": "user", "content": application}]},
        config={"recursion_limit": 20},
    )
    # Show observable tool results, not an assumed private reasoning trace.
    tool_messages = [m for m in result["messages"] if m.type == "tool"]
    expected_tools = {t.name for t in [
        compute_financial_ratios, get_sector_benchmarks, lookup_credit_score_band
    ]}
    missing_tools = expected_tools - {m.name for m in tool_messages}
    failed_tools = [m.name for m in tool_messages if getattr(m, "status", None) == "error"]
    for message in tool_messages:
        print(f"\nTool: {message.name}\n{message.content}")
    if missing_tools or failed_tools:
        print(f"\nEscalate: incomplete tool checks; missing={missing_tools}, failed={failed_tools}")
    else:
        print("\nCredit Assessment (teaching recommendation; requires human review):")
        print(result["messages"][-1].content)
